In [73]:
import os, json
import librosa
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2FeatureExtractor
)
import torch.nn.functional as F

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

whisper_processor = WhisperProcessor.from_pretrained(
    "models/whisper-small",
    local_files_only=True,
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "models/whisper-small",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    "models/qwen2.5-0.5b",
    local_files_only=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen2.5-0.5b",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

WAV2VEC_PATH = "models/wav2vec2-base-960h"
wav2vec_processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)

wav2vec_model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH, local_files_only=True).to(device).eval()

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    WAV2VEC_PATH,
    local_files_only=True
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: models/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [74]:
def continuation_logprob(prefix, continuation):
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    continuation_ids = tokenizer.encode(continuation, add_special_tokens=False)

    if not prefix_ids or not continuation_ids:
        raise ValueError("Prefix and continuation must contain tokens")

    input_ids = torch.tensor(
        [prefix_ids + continuation_ids],
        dtype=torch.long,
        device=device,
    )

    start = len(prefix_ids)

    with torch.inference_mode():
        outputs = llm_model(input_ids=input_ids, use_cache=False)

        logits = outputs.logits[0, start - 1:-1].float()
        targets = input_ids[0, start:]

        token_scores = (logits.gather(1, targets[:, None]).squeeze(1) - torch.logsumexp(logits, dim=-1))

        return token_scores.sum().item()

SYSTEM_PROMPT = "A conversation between two people.\n"

def build_transition_scores(texts, speakers, alpha=0.75):
    ids = sorted(texts)

    lines = {
        i: f"{'A' if speakers[i] == 0 else 'B'}: {texts[i]}\n"
        for i in ids
    }

    neutral_scores = {
        j: continuation_logprob(SYSTEM_PROMPT, lines[j])
        for j in ids
    }

    transitions = {}

    for i in ids:
        for j in ids:
            if speakers[i] == speakers[j]:
                continue

            conditional = continuation_logprob(SYSTEM_PROMPT + lines[i],lines[j],)

            transitions[i, j] = (conditional - alpha * neutral_scores[j])

    return transitions

In [75]:
def beam_orders(
    chunk_ids,
    first,
    second,
    speakers,
    transitions,
    beam_width=200,
    top_k=8,
):
    if beam_width < 1 or top_k < 1:
        raise ValueError("beam_width and top_k must be positive")
    beam_width = max(beam_width, top_k)
    chunk_ids = sorted(chunk_ids)

    bits = {
        chunk_id: 1 << position
        for position, chunk_id in enumerate(chunk_ids)
    }

    initial_mask = bits[first] | bits[second]

    # Переход first -> second одинаков у всех вариантов.
    beam = [(0.0, (first, second), initial_mask)]

    for position in range(2, len(chunk_ids)):
        expected_speaker = position % 2
        merged = {}

        for score, sequence, mask in beam:
            last = sequence[-1]

            for candidate in chunk_ids:
                if mask & bits[candidate]:
                    continue

                if speakers[candidate] != expected_speaker:
                    continue

                new_mask = mask | bits[candidate]
                new_score = score + transitions[last, candidate]
                new_sequence = sequence + (candidate,)

                key = (new_mask, candidate)
                # Keep alternative histories for full-context reranking.
                bucket = merged.setdefault(key, [])
                bucket.append((new_score, new_sequence, new_mask))
                bucket.sort(key=lambda item: item[0], reverse=True)
                del bucket[top_k:]

        beam = sorted(
            (state for bucket in merged.values() for state in bucket),
            key=lambda item: item[0],
            reverse=True,
        )[:beam_width]

        if not beam:
            raise ValueError("No valid order: check speaker assignments")

    return [
        list(sequence)
        for _, sequence, _ in beam[:top_k]
    ]

In [76]:
def full_dialogue_score(sequence, texts, speakers):
    lines = [
        f"{'A' if speakers[i] == 0 else 'B'}: {texts[i]}\n"
        for i in sequence
    ]

    if len(lines) <= 2:
        return 0.0

    return continuation_logprob(
        SYSTEM_PROMPT + "".join(lines[:2]),
        "".join(lines[2:]),
    )


def reconstruct_order(
    transcripts,
    first,
    second,
    speakers,
    alpha=0.75,
    beam_width=200,
    top_k=8,
):
    texts = {
        item["id"]: item["text"]
        for item in transcripts
    }

    assert set(texts) == set(speakers)
    assert speakers[first] == 0
    assert speakers[second] == 1

    transitions = build_transition_scores(
        texts,
        speakers,
        alpha=alpha,
    )

    candidates = beam_orders(
        texts.keys(),
        first,
        second,
        speakers,
        transitions,
        beam_width=beam_width,
        top_k=top_k,
    )

    reranked = [
        {
            "sequence": sequence,
            "score": full_dialogue_score(
                sequence,
                texts,
                speakers,
            ),
        }
        for sequence in candidates
    ]

    reranked.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    return reranked[0]["sequence"], reranked

In [77]:
def score_next_candidate(context, candidate):
    prompt = ("The following is a conversation between two people:\n" + "\n".join(context) + "\nNext turn:")

    context_ids = tokenizer.encode(prompt, add_special_tokens=False)
    candidate_ids = tokenizer.encode(" " + candidate, add_special_tokens=False)

    if not candidate_ids:
        return float("-inf")

    input_ids = torch.tensor(
        [context_ids + candidate_ids],
        dtype=torch.long,
        device=device
    )

    start = len(context_ids)

    with torch.inference_mode():
        outputs = llm_model(input_ids=input_ids)

        logits = outputs.logits[0, start - 1:-1, :]
        targets = input_ids[0, start:]

        log_probs = F.log_softmax(logits.float(), dim=-1)

        token_scores = log_probs.gather(
            dim=-1,
            index=targets.unsqueeze(-1)
        ).squeeze(-1)

        return token_scores.mean().item()

In [78]:
# Reference greedy method for the train comparison only.
def reconstruct_order_greedy(transcripts, first, second):
    texts = {item["id"]: item["text"] for item in transcripts}
    sequence = [first, second]
    remaining = sorted(set(texts) - set(sequence))
    while remaining:
        context = [texts[i] for i in sequence]
        best = max(remaining, key=lambda i: score_next_candidate(context, texts[i]))
        sequence.append(best)
        remaining.remove(best)
    return sequence


In [79]:
def speaker_embedding(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
        duration=12
    )

    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.inference_mode():
        outputs = wav2vec_model(
            inputs.input_values.to(device),
            output_hidden_states=True
        )

    hidden = outputs.hidden_states[6][0]

    embedding = hidden.mean(dim=0)

    embedding = F.normalize(
        embedding.float(),
        dim=0
    )

    return embedding.cpu().numpy()

In [80]:
from pathlib import Path
import numpy as np

a = speaker_embedding(
    Path("dataset/train/0/chunk_0.wav")
)

b = speaker_embedding(
    Path("dataset/train/0/chunk_1.wav")
)

similarity = np.dot(a, b)

print("Similarity:", similarity)

Similarity: 0.65990317


In [81]:
def trans_audio(audio_path):
    au, _ = librosa.load(audio_path, sr=16000)

    features = whisper_processor(au, sampling_rate=16000, return_tensors="pt",).input_features.to(device=device, dtype=next(whisper_model.parameters()).dtype)

    with torch.inference_mode():
        ids = whisper_model.generate(features, language="en", task="transcribe", max_new_tokens=128)

    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

In [82]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

def voice_features(audio_path):
    audio, sr = librosa.load(str(audio_path), sr=16000)

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)

    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

def identify_speakers(files, first, second, iterations=20):
    chunk_ids = [
        int(file.stem.split("_")[1])
        for file in files
    ]

    features = np.stack([
        voice_features(file)
        for file in files
    ])

    first_index = chunk_ids.index(first)
    second_index = chunk_ids.index(second)

    n = len(files)
    speaker_a_count = (n + 1) // 2

    centers = features[[first_index, second_index]].copy()
    free = [
        i for i in range(n)
        if i not in (first_index, second_index)
    ]

    previous = None

    for _ in range(iterations):
        distances = ((features[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)

        ranked = sorted(free,
            key=lambda i: (
                distances[i, 0] - distances[i, 1],
                chunk_ids[i],
            ),
        )

        labels = np.ones(n, dtype=int)
        labels[first_index] = 0
        labels[ranked[:speaker_a_count - 1]] = 0
        labels[second_index] = 1

        if previous is not None and np.array_equal(labels, previous):
            break

        previous = labels.copy()
        centers = np.stack([features[labels == speaker].mean(axis=0) for speaker in (0, 1)])

    return {
        chunk_id: int(label)
        for chunk_id, label in zip(chunk_ids, labels)
    }

In [83]:
import json
from pathlib import Path

SPLIT = "test_leaderboard_a"
TEST_DIR = f"hf-data/private/{SPLIT}"
OUTPUT = f"answers_{SPLIT}.json"

prefixes = json.loads(
    (Path(TEST_DIR) / "prefix.json").read_text()
)

assert len(prefixes) == 100
assert all((Path(TEST_DIR) / did).is_dir() for did in prefixes)

print("Набор:", SPLIT)
print("Диалогов:", len(prefixes))
print("Файл для предсказаний:", OUTPUT)

Набор: test_leaderboard_a
Диалогов: 100
Файл для предсказаний: answers_test_leaderboard_a.json


In [84]:
# Small end-to-end check; does not overwrite predictions.
import inspect
import math
import time
from pathlib import Path

print("reconstruct_order:", inspect.signature(reconstruct_order))
print("identify_speakers:", inspect.signature(identify_speakers))
smoke_started = time.perf_counter()
smoke_id, (smoke_first, smoke_second) = next(iter(prefixes.items()))
smoke_files = sorted(
    (Path(TEST_DIR) / smoke_id).glob("chunk_*.wav"),
    key=lambda path: int(path.stem.split("_")[1]),
)
smoke_ids = [int(path.stem.split("_")[1]) for path in smoke_files]
assert smoke_ids == list(range(len(smoke_files)))

# Reuse transcripts left by an interrupted run only for this same dialogue.
if (
    globals().get("did") == smoke_id
    and isinstance(globals().get("transcripts"), list)
    and [item["id"] for item in transcripts] == smoke_ids
):
    smoke_transcripts = [dict(item) for item in transcripts]
else:
    smoke_transcripts = [
        {"id": i, "text": trans_audio(str(path))}
        for i, path in zip(smoke_ids, smoke_files)
    ]

smoke_features = voice_features(smoke_files[0])
assert smoke_features.shape == (40,) and np.isfinite(smoke_features).all()
smoke_speakers = identify_speakers(smoke_files, smoke_first, smoke_second)
assert smoke_speakers[smoke_first] == 0
assert smoke_speakers[smoke_second] == 1
assert sum(s == 0 for s in smoke_speakers.values()) == (len(smoke_ids) + 1) // 2

# Exercise the ASR path on one real audio chunk.
smoke_asr = trans_audio(str(smoke_files[0]))
assert isinstance(smoke_asr, str)
smoke_sequence, smoke_candidates = reconstruct_order(
    smoke_transcripts, smoke_first, smoke_second, speakers=smoke_speakers
)
assert sorted(smoke_sequence) == smoke_ids
assert smoke_sequence[:2] == [smoke_first, smoke_second]
assert all(smoke_speakers[i] == pos % 2 for pos, i in enumerate(smoke_sequence))
assert all(math.isfinite(item["score"]) for item in smoke_candidates)
assert len({tuple(item["sequence"]) for item in smoke_candidates}) == len(smoke_candidates)

smoke_ranks = [0] * len(smoke_ids)
for pos, i in enumerate(smoke_sequence):
    smoke_ranks[i] = pos
json.dumps({"ranks": smoke_ranks, "speakers": smoke_speakers, "candidates": smoke_candidates})
print("PASS:", smoke_id, "chunks:", len(smoke_ids), "candidates:", len(smoke_candidates))
print("Sequence:", smoke_sequence)
print("Elapsed:", round(time.perf_counter() - smoke_started, 2), "seconds")


reconstruct_order: (transcripts, first, second, speakers, alpha=0.75, beam_width=200, top_k=8)
identify_speakers: (files, first, second, iterations=20)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PASS: 691 chunks: 7 candidates: 8
Sequence: [3, 0, 1, 6, 5, 4, 2]
Elapsed: 0.67 seconds


In [85]:
# Final
import os, json
import time
from pathlib import Path

answers = {}
experiment_details = {}

start_time = time.perf_counter()

for did, (first, second) in prefixes.items():
    folder = Path(TEST_DIR) / did

    files = sorted(folder.glob("chunk_*.wav"), key=lambda p: int(p.stem.split("_")[1]),)

    n = len(files)
    transcripts = []

    for file in files:
        chunk_id = int(file.stem.split("_")[1])

        text = trans_audio(str(file))

        transcripts.append({
            "id": chunk_id,
            "text": text,
        })

    speakers = identify_speakers(files, first, second)

    sequence, reranked = reconstruct_order(transcripts, first, second, speakers=speakers)

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    answers[did] = order

    experiment_details[did] = {
        "transcripts": transcripts,
        "speakers": speakers,
        "sequence": sequence,
        "reranked": reranked,
    }

    print(f"\nDialogue {did}: {sequence}")

    for candidate in reranked:
        print(
            f'  score={candidate["score"]:.3f} '
            f'order={candidate["sequence"]}'
        )

with open(OUTPUT, "w") as f:
    json.dump(answers, f)

with open("qwen_scoring_details.json", "w") as f:
    json.dump(
        experiment_details,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nFinished!")
print(f"Dialogues processed: {len(answers)}")
print(f"Answers: {OUTPUT}")
print("Details: qwen_scoring_details.json")
print(f"Elapsed: {time.perf_counter() - start_time:.1f}s")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 691: [3, 0, 1, 6, 5, 4, 2]
  score=-125.058 order=[3, 0, 1, 6, 5, 4, 2]
  score=-127.147 order=[3, 0, 1, 4, 5, 6, 2]
  score=-128.778 order=[3, 0, 2, 6, 1, 4, 5]
  score=-129.702 order=[3, 0, 2, 6, 5, 4, 1]
  score=-129.861 order=[3, 0, 5, 4, 1, 6, 2]
  score=-130.351 order=[3, 0, 1, 6, 2, 4, 5]
  score=-133.735 order=[3, 0, 5, 4, 2, 6, 1]
  score=-135.001 order=[3, 0, 1, 4, 2, 6, 5]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 895: [0, 1, 2, 6, 3, 4, 5]
  score=-182.682 order=[0, 1, 2, 6, 3, 4, 5]
  score=-186.793 order=[0, 1, 2, 4, 5, 6, 3]
  score=-188.710 order=[0, 1, 3, 6, 2, 4, 5]
  score=-188.737 order=[0, 1, 2, 4, 3, 6, 5]
  score=-189.116 order=[0, 1, 3, 4, 2, 6, 5]
  score=-190.336 order=[0, 1, 2, 6, 5, 4, 3]
  score=-191.023 order=[0, 1, 5, 4, 2, 6, 3]
  score=-193.049 order=[0, 1, 3, 6, 5, 4, 2]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 305: [2, 4, 1, 3, 5, 0, 6]
  score=-131.433 order=[2, 4, 1, 3, 5, 0, 6]
  score=-131.544 order=[2, 4, 5, 0, 6, 3, 1]
  score=-133.979 order=[2, 4, 6, 3, 1, 0, 5]
  score=-134.635 order=[2, 4, 6, 0, 1, 3, 5]
  score=-135.228 order=[2, 4, 6, 3, 5, 0, 1]
  score=-139.051 order=[2, 4, 6, 0, 5, 3, 1]
  score=-142.079 order=[2, 4, 1, 0, 5, 3, 6]
  score=-142.431 order=[2, 4, 1, 0, 6, 3, 5]


... (190 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 940: [5, 10, 7, 3, 6, 9, 4, 14, 11, 18, 0, 1, 2, 16, 8, 15, 19, 17, 13, 12]
  score=-613.669 order=[5, 10, 7, 3, 6, 9, 4, 14, 11, 18, 0, 1, 2, 16, 8, 15, 19, 17, 13, 12]
  score=-614.579 order=[5, 10, 7, 3, 6, 9, 4, 14, 0, 1, 11, 18, 2, 16, 19, 17, 13, 12, 8, 15]
  score=-615.859 order=[5, 10, 7, 3, 6, 9, 4, 14, 0, 1, 11, 18, 2, 16, 8, 15, 19, 17, 13, 12]
  score=-615.917 order=[5, 10, 4, 16, 7, 3, 6, 9, 0, 1, 11, 18, 2, 14, 8, 15, 19, 17, 13, 12]
  score=-616.304 order=[5, 10, 7, 3, 6, 9, 11, 18, 0, 1, 4, 14, 2, 16, 8, 15, 19, 17, 13, 12]
  score=-621.909 order=[5, 10, 4, 16, 7, 3, 6, 9, 11, 14, 0, 1, 2, 18, 8, 15, 19, 17, 13, 12]
  score=-621.930 order=[5, 10, 4, 16, 7, 3, 6, 9, 0, 1, 11, 14, 2, 18, 8, 15, 19, 17, 13, 12]
  score=-623.744 order=[5, 10, 4, 16, 7, 3, 6, 9, 0, 1, 2, 18, 11, 14, 8, 15, 19, 17, 13, 12]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Dialogue 501: [14, 5, 3, 16, 19, 15, 1, 6, 18, 12, 7, 4, 13, 9, 10, 8, 0, 2, 11, 17]
  score=-846.539 order=[14, 5, 3, 16, 19, 15, 1, 6, 18, 12, 7, 4, 13, 9, 10, 8, 0, 2, 11, 17]
  score=-847.895 order=[14, 5, 11, 4, 19, 15, 1, 6, 18, 12, 13, 9, 10, 8, 7, 17, 0, 2, 3, 16]
  score=-850.717 order=[14, 5, 3, 16, 19, 15, 1, 6, 18, 12, 13, 9, 10, 8, 7, 17, 0, 2, 11, 4]
  score=-853.893 order=[14, 5, 3, 16, 19, 15, 1, 6, 18, 12, 10, 8, 7, 4, 13, 9, 0, 2, 11, 17]
  score=-854.308 order=[14, 5, 11, 4, 19, 15, 1, 16, 3, 6, 18, 12, 13, 17, 7, 9, 10, 8, 0, 2]
  score=-854.398 order=[14, 5, 11, 4, 1, 15, 19, 6, 18, 12, 13, 9, 10, 8, 7, 17, 0, 2, 3, 16]
  score=-856.700 order=[14, 5, 11, 4, 19, 15, 1, 16, 3, 6, 18, 12, 13, 17, 7, 9, 0, 2, 10, 8]
  score=-859.865 order=[14, 5, 3, 16, 19, 15, 1, 6, 18, 4, 13, 12, 10, 8, 7, 9, 0, 2, 11, 17]

Finished!
Dialogues processed: 100
Answers: answers_test_leaderboard_a.json
Details: qwen_scoring_details.json
Elapsed: 193.0s


In [86]:
import json
import random
import time
from pathlib import Path

EVAL_DIR = Path("dataset/train")
TRUTH_PATH = Path("hf-data/public/train_answers.json")

N_DIALOGUES = 100
SEED = 42

with open(EVAL_DIR / "prefix.json") as f:
    eval_prefixes = json.load(f)

with open(TRUTH_PATH) as f:
    train_truth = json.load(f)

available_ids = sorted(eval_prefixes, key=int)

assert len(available_ids) >= N_DIALOGUES
assert set(available_ids) <= set(train_truth)

selected_ids = random.Random(SEED).sample(
    available_ids,
    N_DIALOGUES,
)

answers_by_variant = {
    "text_only": {},
    "with_speakers": {},
}

train_experiment_details = {}

start_time = time.perf_counter()

for number, did in enumerate(selected_ids, start=1):
    first, second = eval_prefixes[did]
    folder = EVAL_DIR / did

    files = sorted(
        folder.glob("chunk_*.wav"),
        key=lambda p: int(p.stem.split("_")[1]),
    )

    chunk_ids = [
        int(file.stem.split("_")[1])
        for file in files
    ]

    n = len(files)

    assert chunk_ids == list(range(n)), did
    assert n == len(train_truth[did]), did

    transcripts = []

    for file, chunk_id in zip(files, chunk_ids):
        transcripts.append({
            "id": chunk_id,
            "text": trans_audio(str(file)),
        })

    speakers = identify_speakers(files, first, second)

    # Compare the old text-only greedy method against the new pipeline.
    variants = {
        "text_only": transcripts,
        "with_speakers": transcripts,
    }

    details = {
        "transcripts": transcripts,
        "speakers": speakers,
        "variants": {},
    }

    for name, variant_transcripts in variants.items():
        if name == "text_only":
            sequence = reconstruct_order_greedy(
                variant_transcripts, first, second
            )
            history = []
        else:
            sequence, history = reconstruct_order(
                variant_transcripts, first, second, speakers=speakers
            )

        assert sorted(sequence) == chunk_ids
        assert sequence[:2] == [first, second]

        order = [0] * n

        for pos, chunk_id in enumerate(sequence):
            order[chunk_id] = pos

        answers_by_variant[name][did] = order

        details["variants"][name] = {
            "sequence": sequence,
            "candidates": history,
        }

    train_experiment_details[did] = details

    for name, predictions in answers_by_variant.items():
        with open(f"answers_train100_{name}.json", "w") as f:
            json.dump(predictions, f)

    with open("details_train100.json", "w") as f:
        json.dump(
            {
                "seed": SEED,
                "selected_ids": selected_ids,
                "dialogues": train_experiment_details,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[{number}/{N_DIALOGUES}] Dialogue {did} completed",
        flush=True,
    )

print(f"Elapsed: {time.perf_counter() - start_time:.1f}s")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[1/100] Dialogue 441 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[2/100] Dialogue 89 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[3/100] Dialogue 1096 completed


... (190 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[99/100] Dialogue 1597 completed


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[100/100] Dialogue 1438 completed
Elapsed: 285.9s


In [87]:
def score_dialogue(prediction, target):
    n = len(target)

    if len(prediction) != n:
        return 0.0

    if sorted(prediction) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (prediction[i] - prediction[j])
        * (target[i] - target[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)


baseline_answers = {}

for did in selected_ids:
    first, second = eval_prefixes[did]
    n = len(train_truth[did])

    sequence = [first, second] + [
        i for i in range(n)
        if i not in (first, second)
    ]

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        order[chunk_id] = pos

    baseline_answers[did] = order


all_predictions = {
    "baseline": baseline_answers,
    **answers_by_variant,
}

scores_by_variant = {}

for name, predictions in all_predictions.items():
    assert set(predictions) == set(selected_ids)

    scores = [
        score_dialogue(predictions[did], train_truth[did])
        for did in selected_ids
    ]

    scores_by_variant[name] = np.array(scores)

    print(f"{name:>15}: {np.mean(scores):.6f}")


difference = (
    scores_by_variant["with_speakers"]
    - scores_by_variant["text_only"]
)

print()
print(f"Mean pipeline improvement: {difference.mean():+.6f}")
print(f"Improved: {np.sum(difference > 1e-9)}")
print(f"Worse:    {np.sum(difference < -1e-9)}")
print(f"Equal:    {np.sum(np.abs(difference) <= 1e-9)}")

       baseline: 0.675501
      text_only: 0.725800
  with_speakers: 0.792064

Mean pipeline improvement: +0.066264
Improved: 65
Worse:    32
Equal:    3


In [88]:
# Scoring
import json
from pathlib import Path

pretest_pred = json.loads(Path("answers.json").read_text())
pretest_truth = json.loads(
    Path("hf-data/public/pretest_answers.json").read_text()
)

def score_pretest_dialogue(p, t):
    n = len(t)

    if not isinstance(p, list) or len(p) != n or sorted(p) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (p[i] - p[j]) * (t[i] - t[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)

scores = [
    score_pretest_dialogue(pretest_pred.get(k), pretest_truth[k])
    for k in pretest_truth
]

print(f"Pretest pairwise accuracy: {sum(scores) / len(scores):.6f}")
print(f"Dialogues evaluated: {len(scores)}")

Pretest pairwise accuracy: 0.704561
Dialogues evaluated: 20


In [89]:
score_split = SPLIT
score_output = OUTPUT

predictions = json.loads(Path(score_output).read_text())
targets = json.loads(
    Path(f"hf-data/private/{score_split}_answers.json").read_text()
)

def valid_permutation(p, n):
    return (
        isinstance(p, list)
        and len(p) == n
        and all(type(x) is int for x in p)
        and sorted(p) == list(range(n))
    )

def score_one(prediction, target):
    n = len(target)
    if n < 2:
        return 1.0
    if not valid_permutation(prediction, n):
        return 0.0

    wrong = sum(
        (target[i] > target[j]) != (prediction[i] > prediction[j])
        for i in range(n)
        for j in range(i + 1, n)
    )
    return 1.0 - wrong / (n * (n - 1) // 2)

scores = [
    score_one(predictions.get(did), target)
    for did, target in targets.items()
]

valid = sum(
    valid_permutation(predictions.get(did), len(target))
    for did, target in targets.items()
)

raw = sum(scores) / len(scores)

print("Split:", score_split)
print(f"Valid permutations: {valid}/{len(targets)}")
print(f"Pairwise accuracy: {raw:.8f}")
print(f"Raw accuracy × 100: {round(100 * raw, 4):.4f}")
print("Unused ID:", sorted(set(predictions) - set(targets)))

Split: test_leaderboard_a
Valid permutations: 100/100
Pairwise accuracy: 0.79594799
Raw accuracy × 100: 79.5948
Unused ID: []


In [91]:
# Leaderboard B: independent inference and local scoring.
# Run the model/function definition cells above before this cell.
def evaluate_leaderboard_b():
    import json
    import math
    import time
    from pathlib import Path

    required = ("trans_audio", "identify_speakers", "reconstruct_order")
    missing = [name for name in required if not callable(globals().get(name))]
    if missing:
        raise RuntimeError("Run the definition cells first: " + ", ".join(missing))

    b_dir = Path("hf-data/private/test_leaderboard_b")
    b_output = Path("answers_test_leaderboard_b.json")
    b_details_output = Path("qwen_scoring_details_test_leaderboard_b.json")
    b_truth_path = b_dir.parent / "test_leaderboard_b_answers.json"

    b_prefixes = json.loads((b_dir / "prefix.json").read_text())
    assert len(b_prefixes) == 100, f"Expected 100 dialogues, got {len(b_prefixes)}"

    b_answers = {}
    b_details = {}
    b_started = time.perf_counter()

    for number, (dialogue_id, (first, second)) in enumerate(b_prefixes.items(), 1):
        files = sorted(
            (b_dir / dialogue_id).glob("chunk_*.wav"),
            key=lambda path: int(path.stem.split("_")[1]),
        )
        chunk_ids = [int(path.stem.split("_")[1]) for path in files]
        n = len(files)
        assert n >= 2 and chunk_ids == list(range(n)), dialogue_id
        assert first != second and first in chunk_ids and second in chunk_ids

        transcripts = [
            {"id": chunk_id, "text": trans_audio(str(path))}
            for chunk_id, path in zip(chunk_ids, files)
        ]
        speakers = identify_speakers(files, first, second)
        sequence, candidates = reconstruct_order(
            transcripts, first, second, speakers=speakers
        )

        assert sorted(sequence) == chunk_ids, dialogue_id
        assert sequence[:2] == [first, second], dialogue_id
        assert all(math.isfinite(item["score"]) for item in candidates)

        ranks = [0] * n
        for position, chunk_id in enumerate(sequence):
            ranks[chunk_id] = position

        b_answers[dialogue_id] = ranks
        b_details[dialogue_id] = {
            "transcripts": transcripts,
            "speakers": speakers,
            "sequence": sequence,
            "reranked": candidates,
        }
        print(
            f"B [{number}/{len(b_prefixes)}] dialogue {dialogue_id} "
            f"| elapsed {time.perf_counter() - b_started:.1f}s",
            flush=True,
        )

    b_output.write_text(json.dumps(b_answers), encoding="utf-8")
    b_details_output.write_text(
        json.dumps(b_details, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("Predictions:", b_output.resolve())
    print("Details:", b_details_output.resolve())
    print(f"Elapsed: {time.perf_counter() - b_started:.1f}s")

    # Ground truth is read only after predictions have been produced.
    if b_truth_path.exists():
        targets = json.loads(b_truth_path.read_text())
        assert set(b_answers) == set(targets), "Prediction and target IDs differ"

        scores = []
        for dialogue_id, target in targets.items():
            prediction = b_answers[dialogue_id]
            n = len(target)
            assert len(prediction) == n and sorted(prediction) == list(range(n))
            correct = sum(
                (prediction[i] < prediction[j]) == (target[i] < target[j])
                for i in range(n)
                for j in range(i + 1, n)
            )
            scores.append(correct / (n * (n - 1) // 2) if n > 1 else 1.0)

        raw = sum(scores) / len(scores)
        print(f"Split: test_leaderboard_b | Valid permutations: {len(scores)}/{len(targets)}")
        print(f"Pairwise accuracy: {raw:.8f}")
        print(f"Raw accuracy x 100: {100 * raw:.4f}")
    else:
        print("Ground truth unavailable; predictions saved without scoring.")

    return b_answers

leaderboard_b_answers = evaluate_leaderboard_b()

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

B [1/100] dialogue 732 | elapsed 1.3s


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

B [2/100] dialogue 1027 | elapsed 2.5s


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

B [3/100] dialogue 1930 | elapsed 3.5s


... (190 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

B [99/100] dialogue 1117 | elapsed 178.3s


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

B [100/100] dialogue 278 | elapsed 182.7s
Predictions: /home/kolja/home/kole4ka_bot/first-ml-project/training/ioai/2026/Find_the_Order/answers_test_leaderboard_b.json
Details: /home/kolja/home/kole4ka_bot/first-ml-project/training/ioai/2026/Find_the_Order/qwen_scoring_details_test_leaderboard_b.json
Elapsed: 182.7s
Split: test_leaderboard_b | Valid permutations: 100/100
Pairwise accuracy: 0.78326248
Raw accuracy x 100: 78.3262
